# Cell-table follow-up: neighbour interactions (§14 Step 2) and three checks from `cell-table-map`

Same cells, filters, contact graph and model settings as `cell-table-map` (v1, 2026-10-08). CPU only.

**Fix 1: H3K27me3 beyond nuclear content (Jackson).**
- In `cell-table-map`, H3K27me3/H3 was inflated, because H3 was both in the target and an input.
- Here raw H3K27me3 is predicted, out of fold, from three input sets:
  - nuclear content only (H3, DNA1, DNA2, log area);
  - Danenberg's channels (nuclear + 11 shared);
  - the full Jackson panel.
- The gain over nuclear content is the part of the mark explained by cell identity and state.

**Fix 2: the SMA collapse across cohorts** (r 0.74 within Jackson, 0.02 in Danenberg).
- Correlations of the 14 shared inputs in each cohort, overall and by broad class.
- SMA quantiles by class.
- The SMA model's feature importances.
- The cross-cohort calibration repeated with rank (quantile) harmonisation instead of median / p5–p95.

**Fix 3: what Danenberg's "cd45" channel is.** Its table calls it CD45RO.
- Mean arcsinh signal per published phenotype, next to CD45RA, CD3 and CD20, and the same for Jackson's CD45.
- Pan-CD45 is high on B cells, T cells and macrophages. CD45RO is low on B cells and on naive (CD45RA+) T cells.

**Step 2a: interaction scan.**
- For each cohort, broad class A (epithelial, immune, stromal) and marker M, fitted within type-A cells only:
  - **own:** the cell's other channels + log area;
  - **own+contact mix:** plus the class fractions among touching cells and their count;
  - **own+ring mix:** plus the class fractions among non-touching cells within 30 µm and their count.
- Readouts: ΔR² with a patient-bootstrap CI.
- **Spillover guard:** M is skipped if it is a lineage marker of another class (e.g. no panCK in immune cells).

**Step 2b: pre-specified tests** (fixed before the run).
- Readout: within-image correlation between the out-of-fold residual of M (own model within A cells) and a neighbourhood feature, with a patient-bootstrap CI.
- Danenberg:
  - CD8 T cells: PD-1, ICOS, Ki-67 vs tumour contacts, tumour cells within 30 µm, and distance to the nearest tumour cell;
  - tumour cells: HLA-ABC, β2M, HLA-DR vs T-cell contacts, T cells within 30 µm, and distance to the nearest T cell;
  - macrophages: CD163 vs tumour contacts, tumour cells within 30 µm, and distance to the nearest tumour cell.
- Jackson:
  - **positive control:** tumour CA-IX vs distance to the nearest endothelial cell (expected +; hypoxia grows away from vessels);
  - **hypoxia → H3K27me3:**
    - nuclear-content residual of H3K27me3 in tumour cells vs own CA-IX (expected +);
    - full-model residual vs distance to the nearest endothelial cell (expected +);
    - full-model residual vs distance to the nearest CA-IX-high tumour cell (top 10%; expected −).
- **Spillover flag:** for contact features, we compare with the ring and distance versions. A contact-only effect is flagged.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, kruskal, mannwhitneyu
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
try:
    import lifelines
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifelines"], check=True)
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"          # tiny settings for a local test on synthetic data
SEED = 20261012
COFACTOR = 1.0
MIN_CELLS = 200             # per image, as in earlier runs
MIN_TUM = 50                # tumour cells per patient for patient features
RING_R = 30.0               # µm (= px)
N_FOLD = 5
N_TRAIN = 20_000 if SMOKE else 150_000
N_TRAIN_FINAL = 30_000 if SMOKE else 300_000
N_BOOT = 100 if SMOKE else 1000
N_PERM = 50 if SMOKE else 200
N_REP = 2 if SMOKE else 5
N_JOBS = os.cpu_count()
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=N_JOBS, verbose=-1)
NUC = ["histone_h3", "dna1", "dna2"]
SHARED = {   # Jackson name -> Danenberg name (the 11 shared markers; HER2 is clone 3B5 in both)
    "estrogen_receptor_alpha": "estrogen_receptor_alpha", "c_erb_b_2_her2": "c_erb_b_2_her2_3b5",
    "ki_67": "ki_67", "cd3": "cd3", "cd20": "cd20", "cd45": "cd45", "cd68": "cd68",
    "cytokeratin_5": "cytokeratin_5", "cytokeratin_8_18": "cytokeratin_8_18",
    "pan_cytokeratin": "pan_cytokeratin", "sma": "sma"}
K27, K27R = "histone_h3_trimethylate", "h3k27me3_per_h3"
GROUPS = {
    "danenberg": {
        "clinical": ["estrogen_receptor_alpha", "c_erb_b_2_her2_3b5", "c_erb_b_2_her2_d8f12"],
        "proliferation": ["ki_67"],
        "cell death": ["cleaved_caspase3"],
        "antigen presentation": ["hla_abc", "beta_2_microglobulin", "hla_dr"],
        "immune state": ["cd279_pd_1", "cd278_icos", "cd134", "gitr_tnfrsf18", "cd38", "cd57", "cd45ra"],
        "immune lineage": ["cd45", "cd3", "cd4", "cd8a", "cd20", "cd68", "cd163", "cd11c", "cd15", "cd16", "foxp3"],
        "epithelial lineage": ["pan_cytokeratin", "cytokeratin_8_18", "cytokeratin_5"],
        "stroma and vessels": ["sma", "fsp1", "cd140b_pdgf_receptor_beta", "podoplanin", "caveolin_1", "cd31_v_wf",
                               "cxcl12_sdf_1"]},
    "jackson": {
        "clinical": ["estrogen_receptor_alpha", "progesterone_receptor_a_b", "c_erb_b_2_her2"],
        "proliferation": ["ki_67", "histone_h3_phospho"],
        "cell death": ["cleaved_parp"],
        "epigenetic": [K27, K27R],
        "tumour state": ["s6", "m_tor", "c_myc", "egfr", "p53", "twist", "slug", "cd44", "carbonic_anhydrase_ix"],
        "immune lineage": ["cd45", "cd3", "cd20", "cd68"],
        "epithelial lineage": ["pan_cytokeratin", "cytokeratin_8_18", "cytokeratin_5", "cytokeratin_7",
                               "keratin_14_krt14", "cytokeratin_19", "e_cadherin_p_cadherin", "gata3"],
        "stroma and vessels": ["sma", "vimentin", "fibronectin", "v_wf"]}}
GROUP_OF = {k: {m: g for g, ms in v.items() for m in ms} for k, v in GROUPS.items()}
LITE_TARGETS = ["carbonic_anhydrase_ix", "p53", "c_myc", "s6", "m_tor", "egfr", "twist", "slug", "cd44", K27, K27R,
                "progesterone_receptor_a_b", "gata3", "keratin_14_krt14", "cytokeratin_7", "cytokeratin_19"]
# Expected sign of log HR per SD for DSS, from the literature (verify each before citing). None = no prior.
PRIOR_SIGN = {"carbonic_anhydrase_ix": 1, "p53": 1, "gata3": -1, "progesterone_receptor_a_b": -1, K27: -1, K27R: -1}
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def num(s):
    return pd.to_numeric(pd.Series(s).replace({"nan": np.nan, "[]": np.nan}), errors="coerce")

def near(name, anchor):
    hits = glob.glob(f"{IN}/**/{name}", recursive=True)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def locate():
    out = {}
    for p in sorted(glob.glob(f"{IN}/**/cells.parquet", recursive=True)):
        key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
        assert key not in out, (key, p)
        out[key] = dict(cells=p, report=near("prep_report.json", p), contacts=near("cell_neighbours.parquet", p),
                        clinical=near("clinical.parquet", p))
        if key == "danenberg":
            out[key]["split"] = near("danenberg_patient_split.csv", p)
    print(json.dumps(out, indent=1))
    assert set(out) == {"danenberg", "jackson"}
    return out

In [ ]:
def load(key, P):
    rep = json.load(open(P["report"]))
    chan = list(rep["channel_map"].values()) if key == "danenberg" else list(rep["channel_map"])
    cells = pd.read_parquet(P["cells"])
    print(key, cells.shape, "columns:", list(cells.columns))
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    A = np.arcsinh(cells[chan].values.astype(np.float64) / COFACTOR).astype(np.float32)
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    assert (img_code[ca] == img_code[cb]).all()
    D = dict(key=key, chan=chan, protein=[c for c in chan if c not in NUC], A=A, img=img_code,
             img_names=np.asarray(img_names), starts=starts, patient=cells[pid].astype(str).values,
             broad=broad, epi=epi, xy=cells[["x", "y"]].values.astype(np.float64), ca=ca, cb=cb,
             log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32) if "area" in cells else None,
             cohort=cells["cohort"].astype(str).values if "cohort" in cells else None,
             phen=cells.cell_phenotype.astype(str).values)
    missing = [m for m in D["protein"] if m not in GROUP_OF[key]]
    assert not missing, f"{key}: markers without a group {missing}"
    print(f"{key}: {len(cells):,} cells, {len(img_names)} images, {len(np.unique(D['patient']))} patients, "
          f"{epi.mean():.2f} epithelial, broad {pd.Series(broad).value_counts().to_dict()}, "
          f"{len(ca):,} contact rows mapped, area {'yes' if D['log_area'] is not None else 'no'}")
    return D


def contact_matrix(D):
    n = len(D["A"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def neighbour_means(X, D, C):
    """Mean of X over touching cells (Mc) and over non-touching cells within RING_R of the centroid (Mr)."""
    n, p = X.shape
    Mc = np.full((n, p), np.nan, np.float32)
    Mr = np.full((n, p), np.nan, np.float32)
    nc, nr = np.zeros(n, np.float32), np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr, dc = np.asarray(R.sum(1)).ravel(), np.asarray(Ci.sum(1)).ravel()
        Xi = X[a:b]
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ Xi) / np.maximum(dr, 1)[:, None], np.nan)
        Mc[a:b] = np.where(dc[:, None] > 0, (Ci @ Xi) / np.maximum(dc, 1)[:, None], np.nan)
        nr[a:b], nc[a:b] = dr, dc
    return Mc, Mr, nc, nr

def patient_folds(D):
    pats, pcode = np.unique(D["patient"], return_inverse=True)
    folds = [te for _, te in GroupKFold(N_FOLD).split(np.zeros(len(pcode)), groups=pcode)]
    return folds, pcode, pats

def take(blocks, idx):
    return np.hstack([M[idx][:, cols] for M, cols in blocks])

def fit_model(blocks, y, idx):
    m = lgb.LGBMRegressor(random_state=SEED, **LGB)
    m.fit(take(blocks, idx), y[idx])
    return m

def train_subsets(folds, n, rng):
    """One fixed training subsample per fold, shared by every target and model, so ΔR² reflects features only."""
    allidx = np.arange(n)
    return [np.sort(rng.choice(np.setdiff1d(allidx, te, assume_unique=True),
                               min(N_TRAIN, n - len(te)), replace=False)) for te in folds]

def oof(blocks, y, folds, trains):
    pred = np.empty(len(y), np.float32)
    for te, tr in zip(folds, trains):
        pred[te] = fit_model(blocks, y, tr).predict(take(blocks, te))
    return pred

def pat_stats(y, yhat, pcode, n_pat):
    y, yhat = y.astype(np.float64), yhat.astype(np.float64)
    return (np.bincount(pcode, minlength=n_pat).astype(float), np.bincount(pcode, y, n_pat),
            np.bincount(pcode, y ** 2, n_pat), np.bincount(pcode, (y - yhat) ** 2, n_pat))

def boot_r2(W, s):
    n, sy, syy, sse = s
    point = 1 - sse.sum() / (syy.sum() - sy.sum() ** 2 / n.sum())
    N, SY, SYY, SSE = W @ n, W @ sy, W @ syy, W @ sse
    return point, 1 - SSE / (SYY - SY ** 2 / N)

def ci(v):
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def r_within(y, yhat, img):
    y, yhat = y.astype(np.float64), yhat.astype(np.float64)
    cnt = np.bincount(img)
    yc = y - (np.bincount(img, y) / cnt)[img]
    pc = yhat - (np.bincount(img, yhat) / cnt)[img]
    return float(yc @ pc / np.sqrt((yc @ yc) * (pc @ pc) + 1e-12))

def pearson(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[ok], b[ok])[0, 1])

def signal_quality(y):
    q01, q50, q99 = np.percentile(y, [1, 50, 99])
    mad = 1.4826 * np.median(np.abs(y - q50))
    scale = mad if mad > 1e-3 else y.std() + 1e-6
    return dict(pos_frac=float(np.mean((y - q50) / scale > 3)), dyn_range=float(q99 - q50))

def boot_weights(n_pat, rng):
    return rng.multinomial(n_pat, np.full(n_pat, 1 / n_pat), size=N_BOOT).astype(np.float64)

def harmonise(X):
    med = np.median(X, 0)
    q05, q95 = np.percentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, X.std(0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

## Load both cohorts and shared neighbour structures

In [ ]:
N_TRAIN_SUB = 15_000 if SMOKE else 80_000
PATHS = locate()
DATA = {k: load(k, PATHS[k]) for k in ("danenberg", "jackson")}
CLASSES = ("epithelial", "immune", "stromal")
NB = {}
for key, D in DATA.items():
    C = contact_matrix(D)
    onehot = np.stack([D["broad"] == c for c in CLASSES], 1).astype(np.float32)
    Mc, Mr, nc, nr = neighbour_means(onehot, D, C)
    pats, pcode = np.unique(D["patient"], return_inverse=True)
    NB[key] = dict(C=C, cmix=Mc, rmix=Mr, nc=nc, nr=nr, pats=pats, pcode=pcode,
                   W=boot_weights(len(pats), np.random.default_rng(SEED)))
    tick(f"{key}: class mix of neighbours")
OUT_JSON = {}

def oof_rows(blocks, y, idx, splits):
    pred = np.empty(len(idx), np.float32)
    for tr, te in splits:
        pred[te] = fit_model(blocks, y, idx[tr]).predict(take(blocks, idx[te]))
    return pred

def make_splits(idx, pcode, rng, n_train=N_TRAIN_SUB):
    out = []
    for tr, te in GroupKFold(N_FOLD).split(idx, groups=pcode[idx]):
        out.append((np.sort(rng.choice(tr, min(n_train, len(tr)), replace=False)), te))
    return out

def r2_boot(y, pred, idx, key):
    n = NB[key]
    return boot_r2(n["W"], pat_stats(y[idx], pred, n["pcode"][idx], len(n["pats"])))

## Fix 1: H3K27me3 beyond nuclear content (Jackson)

In [ ]:
DJ, DD = DATA["jackson"], DATA["danenberg"]
cj = {c: i for i, c in enumerate(DJ["chan"])}
rng = np.random.default_rng(SEED + 1)
yK = DJ["A"][:, cj[K27]]
area = [(DJ["log_area"][:, None], [0])] if DJ["log_area"] is not None else []
INPUTS = {"nuclear (H3, DNA1, DNA2, area)": [cj[c] for c in NUC],
          "Danenberg channels (nuclear + 11 shared)": [cj[c] for c in NUC + list(SHARED)],
          "full Jackson panel": [i for i in range(len(DJ["chan"])) if i != cj[K27]]}
rows = []
for subset, mask in (("all cells", np.ones(len(yK), bool)), ("epithelial cells", DJ["epi"])):
    idx = np.flatnonzero(mask)
    splits = make_splits(idx, NB["jackson"]["pcode"], rng)
    R = {k: r2_boot(yK, oof_rows([(DJ["A"], cols)] + area, yK, idx, splits), idx, "jackson") for k, cols in INPUTS.items()}
    base = R["nuclear (H3, DNA1, DNA2, area)"]
    for k, (pt, bv) in R.items():
        rows.append(dict(subset=subset, inputs=k, r2=pt, r2_lo=ci(bv)[0], r2_hi=ci(bv)[1],
                         gain_over_nuclear=pt - base[0], gain_lo=ci(bv - base[1])[0], gain_hi=ci(bv - base[1])[1]))
FIX1 = pd.DataFrame(rows)
FIX1.to_csv(f"{OUT}/fix1_h3k27me3_nuclear_vs_panel.csv", index=False)
print(FIX1.round(3).to_string(index=False))
OUT_JSON["fix1"] = FIX1.round(4).to_dict(orient="records")
tick("fix 1")

## Fix 2: why SMA collapses across cohorts

In [ ]:
IN_J = NUC + list(SHARED)
IN_D = NUC + list(SHARED.values())
XJr = DJ["A"][:, [cj[c] for c in IN_J]]
XDr = DD["A"][:, [DD["chan"].index(c) for c in IN_D]]
corr_rows = []
for key, X, D in (("jackson", XJr, DJ), ("danenberg", XDr, DD)):
    for cls in ("all",) + CLASSES:
        m = np.ones(len(X), bool) if cls == "all" else D["broad"] == cls
        Cm = np.corrcoef(X[m].T)
        for a in range(len(IN_J)):
            for b in range(a + 1, len(IN_J)):
                corr_rows.append(dict(cohort=key, cells=cls, m1=IN_J[a], m2=IN_J[b], r=Cm[a, b]))
CORR = pd.DataFrame(corr_rows)
CORR.to_csv(f"{OUT}/fix2_shared_input_correlations.csv", index=False)
wide = CORR.pivot_table(index=["cells", "m1", "m2"], columns="cohort", values="r").reset_index()
wide["diff"] = wide.danenberg - wide.jackson
print("Largest cohort differences in input correlations involving SMA:")
print(wide[(wide.m1 == "sma") | (wide.m2 == "sma")].sort_values("diff", key=np.abs, ascending=False).head(15).round(3).to_string(index=False))
q_rows = []
for key, X, D in (("jackson", XJr, DJ), ("danenberg", XDr, DD)):
    s = X[:, IN_J.index("sma")]
    for cls in ("all",) + CLASSES:
        m = np.ones(len(s), bool) if cls == "all" else D["broad"] == cls
        q = np.percentile(s[m], [5, 25, 50, 75, 95, 99])
        q_rows.append(dict(cohort=key, cells=cls, n=int(m.sum()), **{f"q{p}": v for p, v in zip((5, 25, 50, 75, 95, 99), q)}))
SMAQ = pd.DataFrame(q_rows)
SMAQ.to_csv(f"{OUT}/fix2_sma_quantiles_arcsinh.csv", index=False)
print(SMAQ.round(3).to_string(index=False))

def rank_norm(X):
    return np.column_stack([(pd.Series(X[:, j]).rank(method="average").values - 0.5) / len(X)
                            for j in range(X.shape[1])]).astype(np.float32)

HARM = {"median / p5-p95 (as in v1)": (harmonise(XJr), harmonise(XDr)), "rank (quantile)": (rank_norm(XJr), rank_norm(XDr))}
cal, imp = [], []
tr = np.sort(rng.choice(len(XJr), min(N_TRAIN_FINAL, len(XJr)), replace=False))
for hname, (XJ, XD) in HARM.items():
    for k, s in enumerate(SHARED):
        j = len(NUC) + k
        cols = [i for i in range(len(IN_J)) if i != j]
        mdl = fit_model([(XJ, cols)], XJ[:, j], tr)
        p = mdl.predict(XD[:, cols]).astype(np.float32)
        cal.append(dict(harmonisation=hname, marker=s, danenberg_r_cell=pearson(XD[:, j], p),
                        danenberg_r_within=r_within(XD[:, j], p, DD["img"])))
        if s == "sma":
            g = mdl.booster_.feature_importance("gain")
            imp += [dict(harmonisation=hname, feature=IN_J[c], gain_share=v / g.sum()) for c, v in zip(cols, g)]
CAL2 = pd.DataFrame(cal)
CAL2.to_csv(f"{OUT}/fix2_calibration_by_harmonisation.csv", index=False)
IMP = pd.DataFrame(imp).sort_values(["harmonisation", "gain_share"], ascending=[True, False])
IMP.to_csv(f"{OUT}/fix2_sma_feature_importance.csv", index=False)
print(CAL2.pivot(index="marker", columns="harmonisation", values="danenberg_r_cell").round(3).to_string())
print(IMP.groupby("harmonisation").head(5).round(3).to_string(index=False))
OUT_JSON["fix2_sma_r_by_harmonisation"] = CAL2[CAL2.marker == "sma"].set_index("harmonisation").danenberg_r_cell.round(3).to_dict()
OUT_JSON["fix2_median_r_by_harmonisation"] = CAL2.groupby("harmonisation").danenberg_r_cell.median().round(3).to_dict()
tick("fix 2")

## Fix 3: what Danenberg's "cd45" channel is

In [ ]:
cd = {c: i for i, c in enumerate(DD["chan"])}
ph = pd.DataFrame({"phenotype": DD["phen"], "broad": DD["broad"]})
for c in ("cd45", "cd45ra", "cd3", "cd20", "cd68", "cd8a", "cd4"):
    ph[c] = DD["A"][:, cd[c]]
TAB = ph.groupby("phenotype").agg(broad=("broad", "first"), n=("cd45", "size"),
                                  **{c: (c, "mean") for c in ("cd45", "cd45ra", "cd3", "cd20", "cd68", "cd8a", "cd4")})
TAB = TAB.sort_values("cd45", ascending=False)
TAB.to_csv(f"{OUT}/fix3_danenberg_cd45_by_phenotype.csv")
print(TAB.round(3).to_string())
phj = pd.DataFrame({"phenotype": DJ["phen"], "cd45": DJ["A"][:, cj["cd45"]], "cd3": DJ["A"][:, cj["cd3"]],
                    "cd20": DJ["A"][:, cj["cd20"]], "cd68": DJ["A"][:, cj["cd68"]]})
TABJ = phj.groupby("phenotype").mean().sort_values("cd45", ascending=False)
TABJ.to_csv(f"{OUT}/fix3_jackson_cd45_by_phenotype.csv")
print(TABJ.round(3).to_string())

def pick(tab, pattern):
    hit = [p for p in tab.index if pattern.lower() in p.lower()]
    return float(tab.loc[hit, "cd45"].mean()) if hit else None

imm = DD["broad"] == "immune"
OUT_JSON["fix3"] = dict(
    danenberg_cd45_B_cells=pick(TAB, "B cells"), danenberg_cd45_CD8_T=pick(TAB, "CD8"),
    danenberg_cd45_macrophages=pick(TAB, "Macrophages"), danenberg_cd45_epithelial_median=float(np.median(DD["A"][DD["epi"], cd["cd45"]])),
    danenberg_r_cd45_vs_cd45ra_immune=pearson(DD["A"][imm, cd["cd45"]], DD["A"][imm, cd["cd45ra"]]),
    jackson_cd45_B_cells=pick(TABJ, "B cell"), jackson_cd45_T_cells=pick(TABJ, "T cell"),
    jackson_cd45_macrophages=pick(TABJ, "Macrophage"))
print(json.dumps(OUT_JSON["fix3"], indent=1))
tick("fix 3")

## Step 2a: interaction scan (within each broad class)

In [ ]:
LINEAGE = {
    "danenberg": {"epithelial": ["pan_cytokeratin", "cytokeratin_8_18", "cytokeratin_5", "estrogen_receptor_alpha",
                                 "c_erb_b_2_her2_3b5", "c_erb_b_2_her2_d8f12"],
                  "immune": ["cd45", "cd45ra", "cd3", "cd4", "cd8a", "cd20", "cd68", "cd163", "cd11c", "cd15", "cd16",
                             "foxp3", "cd38", "cd57"],
                  "stromal": ["sma", "fsp1", "cd140b_pdgf_receptor_beta", "podoplanin", "caveolin_1", "cd31_v_wf"]},
    "jackson": {"epithelial": ["pan_cytokeratin", "cytokeratin_8_18", "cytokeratin_5", "cytokeratin_7", "keratin_14_krt14",
                               "cytokeratin_19", "e_cadherin_p_cadherin", "gata3", "estrogen_receptor_alpha",
                               "progesterone_receptor_a_b", "c_erb_b_2_her2"],
                "immune": ["cd45", "cd3", "cd20", "cd68"],
                "stromal": ["sma", "vimentin", "fibronectin", "v_wf"]}}
SCAN = []
for key, D in DATA.items():
    n = NB[key]
    ci_ = {c: i for i, c in enumerate(D["chan"])}
    extra = [(D["log_area"][:, None], [0])] if D["log_area"] is not None else []
    for A in CLASSES:
        idx = np.flatnonzero(D["broad"] == A)
        if len(idx) < 5000:
            continue
        splits = make_splits(idx, n["pcode"], np.random.default_rng(SEED + 2))
        banned = {m for B, ms in LINEAGE[key].items() if B != A for m in ms}
        for M in [m for m in D["protein"] if m not in banned]:
            y = D["A"][:, ci_[M]]
            cols = [i for i in range(len(D["chan"])) if i != ci_[M]]
            own = [(D["A"], cols)] + extra
            cmix = own + [(n["cmix"], [0, 1, 2]), (n["nc"][:, None], [0])]
            rmix = own + [(n["rmix"], [0, 1, 2]), (n["nr"][:, None], [0])]
            R = {m: r2_boot(y, oof_rows(b, y, idx, splits), idx, key)
                 for m, b in (("own", own), ("contact", cmix), ("ring", rmix))}
            row = dict(cohort=key, cell_class=A, marker=M, group=GROUP_OF[key].get(M, ""), n_cells=len(idx),
                       r2_own=R["own"][0], r2_own_lo=ci(R["own"][1])[0])
            for lab in ("contact", "ring"):
                d = R[lab][1] - R["own"][1]
                row.update({f"dr2_{lab}": R[lab][0] - R["own"][0], f"dr2_{lab}_lo": ci(d)[0], f"dr2_{lab}_hi": ci(d)[1]})
            row["contact_only"] = row["dr2_contact_lo"] > 0 and not row["dr2_ring_lo"] > 0
            SCAN.append(row)
        tick(f"scan {key} {A}: {sum(1 for r in SCAN if r['cohort'] == key and r['cell_class'] == A)} markers")
SCAN = pd.DataFrame(SCAN)
SCAN.to_csv(f"{OUT}/step2_scan.csv", index=False)
print(SCAN.sort_values("dr2_ring", ascending=False).head(25).round(3).to_string(index=False))
OUT_JSON["scan_top_ring"] = SCAN.sort_values("dr2_ring", ascending=False).head(15)[
    ["cohort", "cell_class", "marker", "r2_own", "dr2_ring", "dr2_ring_lo", "dr2_contact", "contact_only"]].round(4).to_dict(orient="records")
OUT_JSON["scan_by_class"] = (SCAN.assign(ring_gain=SCAN.dr2_ring_lo > 0, contact_gain=SCAN.dr2_contact_lo > 0)
                             .groupby(["cohort", "cell_class"])
                             .agg(n_markers=("marker", "size"), n_ring_gain=("ring_gain", "sum"),
                                  n_contact_gain=("contact_gain", "sum"), n_contact_only=("contact_only", "sum"),
                                  max_dr2_ring=("dr2_ring", "max"), median_dr2_ring=("dr2_ring", "median"))
                             .round(4).reset_index().to_dict(orient="records"))

## Step 2b: pre-specified neighbourhood tests

In [ ]:
def count_feature(D, n, mask_B):
    """Contacts with B cells, non-touching B cells within 30 µm, and log1p distance to the nearest other B cell."""
    b = mask_B.astype(np.float32)
    contacts = np.asarray(n["C"] @ b).ravel()
    _, Mr, _, nr = neighbour_means(b[:, None], D, n["C"])
    ring = np.nan_to_num(Mr[:, 0]) * nr
    dist = np.full(len(b), np.nan, np.float32)
    for a, e in zip(D["starts"][:-1], D["starts"][1:]):
        jb = np.flatnonzero(mask_B[a:e])
        if len(jb) < 2:
            continue
        dd, ii = cKDTree(D["xy"][a:e][jb]).query(D["xy"][a:e], k=2)
        self_hit = jb[ii[:, 0]] == np.arange(e - a)
        dist[a:e] = np.log1p(np.where(self_hit, dd[:, 1], dd[:, 0]))
    return {"contacts": contacts, "ring count": ring, "log distance": dist}

def resid_corr(res, f, idx, D, n):
    img, pc = D["img"][idx], n["pcode"][idx]
    ok = np.isfinite(res) & np.isfinite(f)
    res, f, img, pc = res[ok].astype(np.float64), f[ok].astype(np.float64), img[ok], pc[ok]
    k = int(D["img"].max()) + 1
    cnt = np.maximum(np.bincount(img, minlength=k), 1)
    rc = res - (np.bincount(img, res, k) / cnt)[img]
    fc = f - (np.bincount(img, f, k) / cnt)[img]
    P = len(n["pats"])
    sxy, sxx, syy = np.bincount(pc, rc * fc, P), np.bincount(pc, rc * rc, P), np.bincount(pc, fc * fc, P)
    r = sxy.sum() / np.sqrt(sxx.sum() * syy.sum())
    B = (n["W"] @ sxy) / np.sqrt((n["W"] @ sxx) * (n["W"] @ syy))
    return float(r), ci(B), int(ok.sum())

def phen_mask(D, *patterns, exact=False):
    p = pd.Series(D["phen"])
    if exact:
        return p.isin(patterns).values
    return np.any([p.str.contains(x, regex=False).values for x in patterns], 0)

TESTS = []
dk, nD = DATA["danenberg"], NB["danenberg"]
epiD = dk["epi"]
tD = phen_mask(dk, "T cells", "T_{Reg}")
FD = {"tumour": count_feature(dk, nD, epiD), "T cells": count_feature(dk, nD, tD)}
TESTS += [("danenberg", "CD8 T cells", phen_mask(dk, "CD8^{+} T cells", exact=True), m, "full", "tumour", +1)
          for m in ("cd279_pd_1", "cd278_icos", "ki_67")]
TESTS += [("danenberg", "tumour cells", epiD, m, "full", "T cells", +1) for m in ("hla_abc", "beta_2_microglobulin", "hla_dr")]
TESTS += [("danenberg", "macrophages", phen_mask(dk, "Macrophages", exact=True), "cd163", "full", "tumour", +1)]
jk, nJ = DATA["jackson"], NB["jackson"]
epiJ = jk["epi"]
endoJ = phen_mask(jk, "Endothelial")
ca = jk["A"][:, cj["carbonic_anhydrase_ix"]]
caHigh = epiJ & (ca >= np.percentile(ca[epiJ], 90))
FJ = {"endothelial": count_feature(jk, nJ, endoJ), "CA-IX-high tumour": count_feature(jk, nJ, caHigh),
      "own CA-IX": {"value": ca}}
# expected sign is for "more / closer B"; distance features get the opposite sign
TESTS += [("jackson", "tumour cells", epiJ, "carbonic_anhydrase_ix", "full", "endothelial", -1),   # positive control
          ("jackson", "tumour cells", epiJ, K27, "nuclear", "own CA-IX", +1),
          ("jackson", "tumour cells", epiJ, K27, "full", "endothelial", -1),
          ("jackson", "tumour cells", epiJ, K27, "full", "CA-IX-high tumour", +1)]
FEATS = {"danenberg": FD, "jackson": FJ}
pre_rows = []
for key, cellname, maskA, M, model, Bname, expect in TESTS:
    D, n = DATA[key], NB[key]
    ci_ = {c: i for i, c in enumerate(D["chan"])}
    idx = np.flatnonzero(maskA)
    y = D["A"][:, ci_[M]]
    extra = [(D["log_area"][:, None], [0])] if D["log_area"] is not None else []
    cols = [ci_[c] for c in NUC] if model == "nuclear" else [i for i in range(len(D["chan"])) if i != ci_[M]]
    pred = oof_rows([(D["A"], cols)] + extra, y, idx, make_splits(idx, n["pcode"], np.random.default_rng(SEED + 3)))
    res = y[idx] - pred
    for fname, f in FEATS[key][Bname].items():
        r, (lo, hi), n_used = resid_corr(res, f[idx], idx, D, n)
        pre_rows.append(dict(cohort=key, cells=cellname, n_cells=n_used, marker=M, residual_of=model, neighbour=Bname,
                             feature=fname, expected_sign=expect if fname != "log distance" else -expect,
                             r=r, lo=lo, hi=hi, ci_excludes_0=bool(lo > 0 or hi < 0)))
    tick(f"pre-specified {key} {cellname} {M} vs {Bname}")
PRE = pd.DataFrame(pre_rows)
PRE["matches_expected"] = np.where(PRE.ci_excludes_0, np.sign(PRE.r) == PRE.expected_sign, np.nan)
PRE.to_csv(f"{OUT}/step2_prespecified.csv", index=False)
print(PRE.round(3).to_string(index=False))
OUT_JSON["prespecified"] = PRE.round(4).to_dict(orient="records")

## Figures and summary

In [ ]:
COL = {"danenberg": "#41ab5d", "jackson": "#4292c6"}
NAME = {"danenberg": "Danenberg 2022", "jackson": "Jackson 2020"}
fig, ax = plt.subplots(figsize=(8, 0.32 * len(PRE) + 1.5))
y0 = np.arange(len(PRE))[::-1]
for key in DATA:
    m = (PRE.cohort == key).values
    ax.errorbar(PRE.r[m], y0[m], xerr=[PRE.r[m] - PRE.lo[m], PRE.hi[m] - PRE.r[m]], fmt="o", ms=4, lw=1,
                color=COL[key], label=NAME[key])
ax.axvline(0, color="grey", lw=0.6)
ax.set_yticks(y0)
ax.set_yticklabels([f"{r.cells}: {r.marker} ({r.residual_of} residual) ~ {r.neighbour} {r.feature}" for r in PRE.itertuples()],
                   fontsize=7)
ax.set_xlabel("within-image r between residual and neighbourhood feature (95% patient-bootstrap CI)")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), fontsize=7, frameon=False, ncol=2)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{OUT}/fig_prespecified.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
for ax, key in zip(axes, ("danenberg", "jackson")):
    g = SCAN[SCAN.cohort == key].reset_index(drop=True)
    x = np.arange(len(g))
    for lab, w, shade in (("ring", -0.2, 0.55), ("contact", 0.2, 1.0)):
        v = g[f"dr2_{lab}"].values
        ax.bar(x + w, v, 0.4, color=COL[key], alpha=shade, yerr=[v - g[f"dr2_{lab}_lo"], g[f"dr2_{lab}_hi"] - v],
               capsize=1, error_kw=dict(lw=0.5), label=f"{NAME[key]}: ΔR², {'non-touching mix within 30 µm' if lab == 'ring' else 'touching-cell mix'}")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r.cell_class[:3]}: {r.marker}" for r in g.itertuples()], rotation=75, ha="right", fontsize=6)
    ax.set_xlim(-0.6, len(g) - 0.4)
    ax.set_ylabel(f"{NAME[key]}\nΔ R² within cell class\nwhiskers: 95% patient-bootstrap CI", fontsize=8)
    ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1), fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{OUT}/fig_step2_scan.png", dpi=160, bbox_inches="tight")
plt.close(fig)
json.dump(OUT_JSON, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
tick("done")